# BRIDGE-ISL v2 — More Data, Richer Diagnostics, Production Bundle (Kaggle)

**What this notebook does:**

1. Authenticates, discovers the real iSign dataset structure, builds a `video_id`-independent split (no `signer_id` exists in this dataset).
2. Streams `.pose` files from the remote ZIP (no ~158 GB download), extracts Hands+Pose, interpolates gaps, normalizes.
3. **Extracts more data** (configurable `EXTRACT_LIMIT`, resumable) and **saves the landmark dataset** as a reusable artifact.
4. Trains a Transformer encoder-decoder with an LR scheduler, early stopping, best-checkpoint tracking and a CSV training log.
5. **More visualizations:** training/LR curves, sequence-length distribution, vocabulary frequency, BLEU, length calibration, decoder cross-attention heatmap, token-embedding PCA.
6. Packages a **production bundle**: weights + vocab + architecture config + metrics + plots + an `ISLTranslator` inference class.

**Kaggle setup:** accept the dataset license at https://huggingface.co/datasets/Exploration-Lab/iSign; Settings → Accelerator GPU, Internet **On**; add Kaggle Secret `HF_TOKEN` (toggled on for this notebook).

**Honest framing of "production":** this gives you a clean, versioned, reloadable artifact and a tidy inference API. It does not make the model accurate by itself — quality depends on how much data you extract (Step 5) and how long you train. Check the BLEU and sample generations in Step 9 before trusting any output.


In [ ]:
!pip install -q "huggingface_hub>=0.24,<1.0" pose-format pyyaml tqdm sacrebleu matplotlib scikit-learn


In [ ]:
import os

IS_KAGGLE = os.path.exists("/kaggle/input")
hf_token = None

if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    try:
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        print("Loaded HF_TOKEN from Kaggle Secrets.")
    except Exception as e:
        raise RuntimeError(
            "Could not load HF_TOKEN from Kaggle Secrets. "
            "Add it under Add-ons -> Secrets (label exactly HF_TOKEN), "
            "and make sure it's toggled ON for this notebook."
        ) from e
    from huggingface_hub import login
    login(token=hf_token)
else:
    # Colab / local fallback: interactive login
    from huggingface_hub import notebook_login
    notebook_login()


In [ ]:
import os
for d in ["src/data", "src/landmarks", "src/inference", "scripts", "configs", "data", "checkpoints", "artifacts"]:
    os.makedirs(d, exist_ok=True)
for p in ["src", "src/data", "src/landmarks", "src/inference", "scripts"]:
    open(f"{p}/__init__.py", "w").close()
print("Working directory:", os.getcwd())


## Step 1 — Config

In [ ]:
%%writefile configs/data.yaml
dataset:
  repo_id: "Exploration-Lab/iSign"
  pose_parts:
    - "iSign-poses_v1.1_part_aa"
    - "iSign-poses_v1.1_part_ab"
    - "iSign-poses_v1.1_part_ac"
    - "iSign-poses_v1.1_part_ad"
  pose_archive_prefix: "iSign-poses_v1.1"

split:
  val_fraction: 0.1
  test_fraction: 0.1
  seed: 42

features:
  components:
    - "POSE_LANDMARKS"
    - "LEFT_HAND_LANDMARKS"
    - "RIGHT_HAND_LANDMARKS"

paths:
  raw_csv: "data/iSign_v1.1.csv"
  metadata_csv: "data/metadata.csv"
  processed_dir: "data/processed"


## Step 2 — Discover the real repo files and download the CSV

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

REPO_ID = "Exploration-Lab/iSign"
api = HfApi()
files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
print(f"{len(files)} files found:")
for f in sorted(files):
    print(" -", f)


In [ ]:
csv_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset",
                            filename="iSign_v1.1.csv", local_dir="data")
print("Downloaded:", csv_path)

import pandas as pd
df_check = pd.read_csv(csv_path)
print(df_check.shape, df_check.columns.tolist())
df_check.head()


## Step 3 — Landmark modules (remote ZIP streaming + preprocessing)

In [ ]:
%%writefile src/landmarks/remote_zip.py
"""
Streams individual files out of the multi-part iSign pose ZIP archive
hosted on Hugging Face, without downloading the full (~158 GB) archive.

Confirmed by direct inspection:
  - iSign-poses_v1.1_part_aa..ad are byte-splits of one valid ZIP file.
  - Each entry is named '{uid}.pose' inside 'iSign-poses_v1.1/'.
  - zipfile only needs the central directory (near the end of the last
    part) to list and random-access any single member.
"""
import requests
from huggingface_hub import hf_hub_url


class RemoteMultipartZip:
    def __init__(self, repo_id, part_filenames, repo_type="dataset", token=None):
        self.token = token
        self.session = requests.Session()
        self.parts = []
        offset = 0
        for fname in part_filenames:
            url = hf_hub_url(repo_id=repo_id, filename=fname, repo_type=repo_type)
            r = self.session.get(url, headers=self._auth_headers() | {"Range": "bytes=0-0"})
            r.raise_for_status()
            total = int(r.headers["Content-Range"].split("/")[-1])
            self.parts.append({"url": url, "start": offset, "end": offset + total, "size": total})
            offset += total
        self.total_size = offset
        self._pos = 0

    def _auth_headers(self):
        return {"Authorization": f"Bearer {self.token}"} if self.token else {}

    def seekable(self):
        return True

    def seek(self, offset, whence=0):
        if whence == 0:
            self._pos = offset
        elif whence == 1:
            self._pos += offset
        elif whence == 2:
            self._pos = self.total_size + offset
        else:
            raise ValueError(f"Unsupported whence: {whence}")
        return self._pos

    def tell(self):
        return self._pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.total_size - self._pos
        end_pos = min(self._pos + n, self.total_size)
        out = bytearray()
        pos = self._pos
        while pos < end_pos:
            part = next(p for p in self.parts if p["start"] <= pos < p["end"])
            lo = pos - part["start"]
            hi = min(end_pos - part["start"], part["size"]) - 1
            r = self.session.get(part["url"],
                                  headers=self._auth_headers() | {"Range": f"bytes={lo}-{hi}"})
            r.raise_for_status()
            out.extend(r.content)
            pos = part["start"] + hi + 1
        self._pos = end_pos
        return bytes(out)


In [ ]:
%%writefile src/landmarks/pose_features.py
"""
Feature selection and preprocessing for iSign .pose files.

Real header (confirmed by inspection; 576 points, confidence stored separately):
POSE_LANDMARKS 33, FACE_LANDMARKS 468, LEFT_HAND_LANDMARKS 21,
RIGHT_HAND_LANDMARKS 21, POSE_WORLD_LANDMARKS 33.

Point indices are resolved from each file's own header, never hardcoded.
pose-format returns MASKED arrays; np.save fails on them (NotImplementedError
in tofile), so everything is converted to plain float32 arrays on slicing.
"""
import numpy as np


def _to_plain_array(arr, fill_value=0.0):
    if np.ma.isMaskedArray(arr):
        return np.ma.filled(arr, fill_value).astype(np.float32)
    return np.asarray(arr, dtype=np.float32)


def component_slices(header):
    slices, offset = {}, 0
    for component in header.components:
        n = len(component.points)
        slices[component.name] = (offset, offset + n)
        offset += n
    return slices


def select_components(pose, component_names):
    slices = component_slices(pose.header)
    missing = [c for c in component_names if c not in slices]
    if missing:
        raise ValueError(f"Components {missing} not in header. Available: {list(slices)}")

    name_to_component = {c.name: c for c in pose.header.components}
    data_chunks, conf_chunks, point_names = [], [], []
    for name in component_names:
        start, end = slices[name]
        data_chunks.append(_to_plain_array(pose.body.data[:, :, start:end, :]))
        conf_chunks.append(_to_plain_array(pose.body.confidence[:, :, start:end]))
        point_names.extend([f"{name}:{p}" for p in name_to_component[name].points])

    return np.concatenate(data_chunks, axis=2), np.concatenate(conf_chunks, axis=2), point_names


def find_point_index(point_names, component_name, candidates):
    for cand in candidates:
        target = f"{component_name}:{cand}"
        if target in point_names:
            return point_names.index(target)
    raise ValueError(
        f"None of {candidates} under {component_name}. "
        f"Available: {[p for p in point_names if p.startswith(component_name)]}"
    )


def interpolate_missing(data, confidence, zero_conf_threshold=1e-6):
    frames, people, points, dims = data.shape
    out = data.copy()
    frame_idx = np.arange(frames)
    for p in range(people):
        for k in range(points):
            valid = confidence[:, p, k] > zero_conf_threshold
            if valid.all() or not valid.any():
                continue
            for d in range(dims):
                out[:, p, k, d] = np.interp(frame_idx, frame_idx[valid], data[valid, p, k, d])
    return out


def normalize_sequence(data, point_names, pose_component="POSE_LANDMARKS"):
    left_idx = find_point_index(point_names, pose_component, ["LEFT_SHOULDER"])
    right_idx = find_point_index(point_names, pose_component, ["RIGHT_SHOULDER"])

    left, right = data[:, 0, left_idx, :2], data[:, 0, right_idx, :2]
    origin = (left + right) / 2.0
    scale = np.linalg.norm(left - right, axis=-1)

    valid_scale = scale[scale > 1e-6]
    median_scale = float(np.median(valid_scale)) if valid_scale.size else 1.0
    scale = np.maximum(np.where(scale > 1e-6, scale, median_scale), 1e-6)

    out = data.copy()
    out[:, 0, :, 0] -= origin[:, None, 0]
    out[:, 0, :, 1] -= origin[:, None, 1]
    out[:, 0, :, :] /= scale[:, None, None]
    return out


## Step 4 — Metadata table (video_id-independent split)

In [ ]:
%%writefile src/data/build_metadata.py
"""
Builds the Phase 1 metadata table. No signer_id field exists anywhere in
this dataset (confirmed by direct inspection) -- so video_id is the
correct leakage boundary, matching the dataset card's own recommendation.
"""
import argparse
import random
from pathlib import Path

import pandas as pd
import yaml


def derive_video_id(uid: str) -> str:
    return uid.rsplit("-", 1)[0]


def split_by_video_id(video_ids, val_fraction, test_fraction, seed):
    unique_ids = sorted(set(video_ids))
    rng = random.Random(seed)
    rng.shuffle(unique_ids)

    n = len(unique_ids)
    n_test = int(n * test_fraction)
    n_val = int(n * val_fraction)

    test_ids = set(unique_ids[:n_test])
    val_ids = set(unique_ids[n_test:n_test + n_val])
    train_ids = set(unique_ids[n_test + n_val:])

    assert train_ids.isdisjoint(val_ids)
    assert train_ids.isdisjoint(test_ids)
    assert val_ids.isdisjoint(test_ids)
    return train_ids, val_ids, test_ids


def assign_split(video_id, train_ids, val_ids, test_ids):
    if video_id in train_ids:
        return "train"
    if video_id in val_ids:
        return "val"
    if video_id in test_ids:
        return "test"
    raise ValueError(f"video_id {video_id} was not assigned to any split")


def build_metadata(csv_path, val_fraction, test_fraction, seed):
    df = pd.read_csv(csv_path)
    if not {"uid", "text"}.issubset(df.columns):
        raise ValueError(f"Expected columns 'uid','text' in {csv_path}, got {df.columns.tolist()}.")

    df["video_id"] = df["uid"].apply(derive_video_id)
    train_ids, val_ids, test_ids = split_by_video_id(df["video_id"], val_fraction, test_fraction, seed)
    df["split"] = df["video_id"].apply(lambda v: assign_split(v, train_ids, val_ids, test_ids))

    out = pd.DataFrame({
        "sample_id": df["uid"],
        "uid": df["uid"],
        "video_id": df["video_id"],
        "english_text": df["text"],
        "split": df["split"],
        "sequence_length": pd.NA,
    })

    print(f"Total rows: {len(out)}")
    print(f"Unique videos: {out['video_id'].nunique()}")
    print(out["split"].value_counts())
    return out


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", type=str, required=True)
    args = parser.parse_args()

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    out_df = build_metadata(
        csv_path=cfg["paths"]["raw_csv"],
        val_fraction=cfg["split"]["val_fraction"],
        test_fraction=cfg["split"]["test_fraction"],
        seed=cfg["split"]["seed"],
    )

    out_path = Path(cfg["paths"]["metadata_csv"])
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(out_path, index=False)
    print(f"Wrote metadata to {out_path}")


if __name__ == "__main__":
    main()


In [ ]:
!python -m src.data.build_metadata --config configs/data.yaml


## Step 5 — Extract landmarks (more data)

Measured earlier: ~500 samples in ~7 min (~1.15 samples/sec). Rough budget:

| `EXTRACT_LIMIT` | approx. time |
|---|---|
| 3,000 | ~45 min |
| 8,000 | ~2 h |
| 20,000 | ~5 h |
| full 127,237 | ~30 h (multiple sessions) |

Cached `.npy` files are skipped on rerun and corrupt/empty cache files are detected and redone, so extraction is resumable — **as long as `data/processed/` survives** (Step 5b saves it).

Note: the metadata table is ordered by video, so `--limit N` takes the first N rows (a handful of whole videos per split slice). For a more representative sample, the next cell shuffles by `video_id` before limiting.


In [ ]:
%%writefile scripts/extract_landmarks.py
"""
Streams each needed .pose file remotely, extracts the configured components,
interpolates missing landmarks, normalizes, caches to data/processed/{sample_id}.npy,
and writes sequence_length back into the metadata table. Resumable.
"""
import argparse
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from pose_format import Pose
from tqdm import tqdm

from src.landmarks.remote_zip import RemoteMultipartZip
from src.landmarks.pose_features import select_components, interpolate_missing, normalize_sequence


def load_cached(out_path):
    """Frame count of a valid cached file, else None (corrupt/empty files are deleted)."""
    if not out_path.exists():
        return None
    try:
        arr = np.load(out_path)
        if arr.size == 0:
            raise ValueError("empty array")
        return arr.shape[0]
    except Exception as e:
        print(f"Corrupt cache {out_path.name} ({e!r}) - deleting and reprocessing.")
        out_path.unlink()
        return None


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--token", default=None)
    parser.add_argument("--limit", type=int, default=None)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    with open(args.config) as f:
        cfg = yaml.safe_load(f)

    metadata_path = Path(cfg["paths"]["metadata_csv"])
    processed_dir = Path(cfg["paths"]["processed_dir"])
    processed_dir.mkdir(parents=True, exist_ok=True)
    if not metadata_path.exists():
        raise FileNotFoundError(f"{metadata_path} missing - run build_metadata first.")

    full_df = pd.read_csv(metadata_path)

    if args.limit:
        # Pick whole videos at random (seeded) so the subset spans many videos,
        # keeping each video's segments together (no split leakage is introduced,
        # since the split was assigned per video_id beforehand).
        rng = np.random.RandomState(args.seed)
        video_ids = full_df["video_id"].unique()
        rng.shuffle(video_ids)
        chosen, count = [], 0
        sizes = full_df.groupby("video_id").size()
        for v in video_ids:
            chosen.append(v)
            count += sizes[v]
            if count >= args.limit:
                break
        df = full_df[full_df["video_id"].isin(chosen)].copy()
    else:
        df = full_df.copy()

    print(f"Processing {len(df)} rows from {df['video_id'].nunique()} videos")
    print("Indexing remote pose archive (central directory only)...")
    remote_zip = RemoteMultipartZip(
        repo_id=cfg["dataset"]["repo_id"],
        part_filenames=cfg["dataset"]["pose_parts"],
        token=args.token,
    )
    zf = zipfile.ZipFile(remote_zip)
    available = set(zf.namelist())
    prefix = cfg["dataset"]["pose_archive_prefix"]
    components = cfg["features"]["components"]

    seq_lengths, n_missing, n_failed = {}, 0, 0
    for row in tqdm(df.itertuples(), total=len(df)):
        out_path = processed_dir / f"{row.sample_id}.npy"

        cached = load_cached(out_path)
        if cached is not None:
            seq_lengths[row.sample_id] = cached
            continue

        entry = f"{prefix}/{row.uid}.pose"
        if entry not in available:
            n_missing += 1
            continue
        try:
            pose = Pose.read(zf.read(entry))
            data, conf, point_names = select_components(pose, components)
            data = interpolate_missing(data, conf)
            data = normalize_sequence(data, point_names)
            flat = data[:, 0, :, :].reshape(data.shape[0], -1).astype(np.float32)
            np.save(out_path, flat)
            seq_lengths[row.sample_id] = flat.shape[0]
        except Exception as e:
            print(f"Failed on {row.sample_id}: {e!r}")
            n_failed += 1

    # Merge results into the FULL table so earlier sessions' progress is never lost.
    full_df["sequence_length"] = full_df["sample_id"].map(seq_lengths).combine_first(
        pd.to_numeric(full_df["sequence_length"], errors="coerce")
    )
    full_df.to_csv(metadata_path, index=False)
    print(f"Done. {len(seq_lengths)} cached, {n_missing} missing from archive, {n_failed} failed.")


if __name__ == "__main__":
    main()


In [ ]:
# Scale this up across sessions (see the table above). None = every row.
EXTRACT_LIMIT = 5000

if not IS_KAGGLE:
    from huggingface_hub import HfFolder
    hf_token = HfFolder.get_token()

limit_arg = f"--limit {EXTRACT_LIMIT}" if EXTRACT_LIMIT else ""
!python -m scripts.extract_landmarks --config configs/data.yaml --token {hf_token} {limit_arg}


In [ ]:
import numpy as np
import pandas as pd

meta = pd.read_csv("data/metadata.csv")
extracted = meta.dropna(subset=["sequence_length"])
print(f"{len(extracted)} samples extracted | {extracted['video_id'].nunique()} videos")
print(extracted.groupby("split")["sequence_length"].describe())

sid = extracted.iloc[0]["sample_id"]
arr = np.load(f"data/processed/{sid}.npy")
print(sid, "-> shape:", arr.shape, "(expect frames x 225)")


## Step 5b — Save the extracted dataset

Zips `data/processed/` and `metadata.csv`. After running: **Save Version**, then from the Output tab download the zip or click **New Dataset** to publish it as a reusable Kaggle Dataset. In a future session add it as input, copy/unzip it into `data/`, and skip Steps 2–5.

In [ ]:
import shutil, os

base = "artifacts/bridge_isl_landmarks_dataset"
shutil.make_archive(base, "zip", root_dir="data", base_dir="processed")
shutil.copy("data/metadata.csv", "artifacts/metadata.csv")
print(f"Saved {base}.zip ({os.path.getsize(base + '.zip') / 1e6:.1f} MB), {len(extracted)} samples")

from IPython.display import FileLink
FileLink(base + ".zip")


## Step 6 — Tokenizer and PyTorch Dataset

Vocabulary is built from the **training split only**. Special tokens: `<PAD>`, `<BOS>`, `<EOS>`, `<UNK>`. Word-level tokenization is intentionally simple; a subword tokenizer can replace it later without touching the model.

In [ ]:
import re, json
from collections import Counter

PAD, BOS, EOS, UNK = "<PAD>", "<BOS>", "<EOS>", "<UNK>"
SPECIAL_TOKENS = [PAD, BOS, EOS, UNK]

def tokenize(text):
    return re.findall(r"[a-z0-9']+", str(text).lower())

def build_vocab(texts, min_freq=2):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))
    vocab = list(SPECIAL_TOKENS) + [w for w, f in counter.most_common() if f >= min_freq]
    return {tok: i for i, tok in enumerate(vocab)}, counter

extracted = pd.read_csv("data/metadata.csv").dropna(subset=["sequence_length"])
train_texts = extracted[extracted["split"] == "train"]["english_text"].tolist()

token_to_id, word_counts = build_vocab(train_texts, min_freq=2)
id_to_token = {i: t for t, i in token_to_id.items()}
print("Vocab size:", len(token_to_id))
with open("data/vocab.json", "w") as f:
    json.dump(token_to_id, f)


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class ISLDataset(Dataset):
    def __init__(self, metadata_df, processed_dir, token_to_id, split):
        self.df = metadata_df[(metadata_df["split"] == split) & metadata_df["sequence_length"].notna()].reset_index(drop=True)
        self.processed_dir = processed_dir
        self.token_to_id = token_to_id

    def __len__(self):
        return len(self.df)

    def encode_text(self, text):
        ids = [self.token_to_id[BOS]]
        ids += [self.token_to_id.get(t, self.token_to_id[UNK]) for t in tokenize(text)]
        ids.append(self.token_to_id[EOS])
        return ids

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        landmarks = np.load(f"{self.processed_dir}/{row['sample_id']}.npy")
        return torch.from_numpy(landmarks).float(), torch.tensor(self.encode_text(row["english_text"]), dtype=torch.long)


def collate_fn(batch):
    landmarks, token_ids = zip(*batch)
    max_frames = max(x.shape[0] for x in landmarks)
    feat_dim = landmarks[0].shape[1]
    padded = torch.zeros(len(landmarks), max_frames, feat_dim)
    src_mask = torch.zeros(len(landmarks), max_frames, dtype=torch.bool)  # True = real frame
    for i, x in enumerate(landmarks):
        padded[i, : x.shape[0]] = x
        src_mask[i, : x.shape[0]] = True

    max_tokens = max(t.shape[0] for t in token_ids)
    tokens = torch.full((len(token_ids), max_tokens), token_to_id[PAD], dtype=torch.long)
    for i, t in enumerate(token_ids):
        tokens[i, : t.shape[0]] = t
    return padded, src_mask, tokens


metadata_full = pd.read_csv("data/metadata.csv")
train_ds = ISLDataset(metadata_full, "data/processed", token_to_id, "train")
val_ds = ISLDataset(metadata_full, "data/processed", token_to_id, "val")
print(f"train: {len(train_ds)}  val: {len(val_ds)}")

BATCH_SIZE = 16
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

xb, mb, yb = next(iter(train_loader))
print("batch:", xb.shape, mb.shape, yb.shape)


In [ ]:
import torch.nn as nn
import random as py_random

SEED = 42
py_random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

FEATURE_DIM = 225  # confirmed: 75 points (pose + two hands) x 3
VOCAB_SIZE = len(token_to_id)
MODEL_CONFIG = {
    "feature_dim": FEATURE_DIM, "vocab_size": VOCAB_SIZE,
    "d_model": 256, "nhead": 4,
    "num_encoder_layers": 3, "num_decoder_layers": 3,
    "dim_feedforward": 512, "dropout": 0.1,
}
print(MODEL_CONFIG)


## Step 7 — Transformer encoder-decoder

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, dim, max_len=2000):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, dim, 2).float() * (-np.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class Seq2SeqTransformer(nn.Module):
    def __init__(self, feature_dim, vocab_size, d_model=256, nhead=4, num_encoder_layers=3,
                 num_decoder_layers=3, dim_feedforward=512, dropout=0.1, pad_id=0):
        super().__init__()
        self.src_proj = nn.Linear(feature_dim, d_model)
        self.tgt_embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_enc = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead, num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, src, src_key_padding_mask, tgt_in, tgt_key_padding_mask):
        src_emb = self.pos_enc(self.src_proj(src))
        tgt_emb = self.pos_enc(self.tgt_embed(tgt_in))
        n = tgt_in.size(1)
        causal = torch.triu(torch.ones(n, n, device=src.device) * float("-inf"), diagonal=1)
        out = self.transformer(
            src_emb, tgt_emb,
            src_key_padding_mask=~src_key_padding_mask,
            tgt_key_padding_mask=~tgt_key_padding_mask,
            tgt_mask=causal,
        )
        return self.out(out)


model = Seq2SeqTransformer(pad_id=token_to_id[PAD], **MODEL_CONFIG).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


## Step 8 — Training

Teacher forcing, loss ignores `<PAD>`. Adds: `ReduceLROnPlateau`, early stopping, best-checkpoint saving (`*_best.pt`), and a per-epoch `training_log.csv`.

In [ ]:
import time, csv

def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total, n = 0.0, 0
    with torch.set_grad_enabled(training):
        for src, src_mask, tgt in loader:
            src, src_mask, tgt = src.to(DEVICE), src_mask.to(DEVICE), tgt.to(DEVICE)
            tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
            logits = model(src, src_mask, tgt_in, tgt_in != token_to_id[PAD])
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))
            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total += loss.item(); n += 1
    return total / max(n, 1)


def train_model(model, name, epochs=30, lr=5e-4, patience=5, log_path="artifacts/training_log.csv"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
    criterion = nn.CrossEntropyLoss(ignore_index=token_to_id[PAD])
    history = {"train_loss": [], "val_loss": [], "lr": []}
    best, stale = float("inf"), 0

    with open(log_path, "w", newline="") as f:
        csv.writer(f).writerow(["epoch", "train_loss", "val_loss", "lr", "seconds"])

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr = run_epoch(model, train_loader, criterion, optimizer)
        va = run_epoch(model, val_loader, criterion)
        scheduler.step(va)
        lr_now = optimizer.param_groups[0]["lr"]
        history["train_loss"].append(tr); history["val_loss"].append(va); history["lr"].append(lr_now)
        secs = time.time() - t0
        print(f"[{name}] epoch {epoch}/{epochs} train={tr:.4f} val={va:.4f} lr={lr_now:.1e} ({secs:.1f}s)")
        with open(log_path, "a", newline="") as f:
            csv.writer(f).writerow([epoch, tr, va, lr_now, round(secs, 1)])

        if va < best:
            best, stale = va, 0
            torch.save(model.state_dict(), f"checkpoints/{name}_best.pt")
            print(f"  -> new best, saved {name}_best.pt")
        else:
            stale += 1
            if stale >= patience:
                print(f"  -> no improvement for {patience} epochs, stopping early.")
                break
    return history, best


history, best_val_loss = train_model(model, "isl_transformer", epochs=30, patience=5)
print(f"Best val loss: {best_val_loss:.4f}")
model.load_state_dict(torch.load("checkpoints/isl_transformer_best.pt", map_location=DEVICE))


## Step 9 — Visualizations

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="val")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title("Training curve")
ax[1].plot(history["lr"]); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("learning rate"); ax[1].set_title("LR schedule")
plt.tight_layout(); plt.savefig("artifacts/training_curves.png", dpi=120); plt.show()


**Sequence-length distribution by split** — large differences between splits would signal an unrepresentative sample.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for split in ["train", "val", "test"]:
    s = extracted[extracted["split"] == split]["sequence_length"]
    if len(s):
        ax.hist(s, bins=30, alpha=0.5, label=f"{split} (n={len(s)})")
ax.set_xlabel("frames"); ax.set_ylabel("count"); ax.legend(); ax.set_title("Sequence length by split")
plt.tight_layout(); plt.savefig("artifacts/sequence_length_distribution.png", dpi=120); plt.show()


**Vocabulary frequency** — the long tail is why `min_freq=2` maps rare words to `<UNK>`.

In [ ]:
words, counts = zip(*word_counts.most_common(25))
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(range(len(words)), counts); ax.set_yticks(range(len(words))); ax.set_yticklabels(words); ax.invert_yaxis()
ax.set_xlabel("count in training set")
ax.set_title(f"Top 25 words (vocab {len(token_to_id)}, unique seen {len(word_counts)})")
plt.tight_layout(); plt.savefig("artifacts/vocab_frequency.png", dpi=120); plt.show()


**Generation, BLEU and length calibration.** Note `ISLDataset` returns 2 values `(landmarks, token_ids)`; for a single unpadded sample the mask is all-True.

In [ ]:
import sacrebleu

@torch.no_grad()
def greedy_decode(model, landmarks, max_len=40):
    model.eval()
    src = landmarks.unsqueeze(0).to(DEVICE)
    mask = torch.ones(1, landmarks.shape[0], dtype=torch.bool, device=DEVICE)
    ids = torch.tensor([[token_to_id[BOS]]], device=DEVICE)
    for _ in range(max_len):
        logits = model(src, mask, ids, torch.ones_like(ids, dtype=torch.bool))
        nxt = logits[0, -1].argmax().item()
        ids = torch.cat([ids, torch.tensor([[nxt]], device=DEVICE)], dim=1)
        if nxt == token_to_id[EOS]:
            break
    toks = [id_to_token.get(i, UNK) for i in ids[0].tolist()]
    return " ".join(t for t in toks if t not in SPECIAL_TOKENS)


N_EVAL = min(100, len(val_ds))
refs, hyps = [], []
for i in range(N_EVAL):
    lm, _ = val_ds[i]
    refs.append(val_ds.df.iloc[i]["english_text"])
    hyps.append(greedy_decode(model, lm))

bleu = sacrebleu.corpus_bleu(hyps, [refs])
print(f"Corpus BLEU on {N_EVAL} val examples: {bleu.score:.2f}")
print("\nSamples:")
for r, h in list(zip(refs, hyps))[:6]:
    print("  REF:", r); print("  GEN:", h); print()


In [ ]:
ref_len = [len(tokenize(r)) for r in refs]
hyp_len = [len(h.split()) for h in hyps]
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(ref_len, hyp_len, alpha=0.6)
m = max(max(ref_len, default=1), max(hyp_len, default=1))
ax.plot([0, m], [0, m], "r--", label="perfect")
ax.set_xlabel("reference words"); ax.set_ylabel("generated words"); ax.legend(); ax.set_title("Length calibration")
plt.tight_layout(); plt.savefig("artifacts/length_calibration.png", dpi=120); plt.show()


**Decoder cross-attention** — which input frames the decoder attends to for each output position (teacher-forced pass on one validation example).

In [ ]:
captured = {}

def _hook(module, inputs, output):
    captured["w"] = output[1]

# need_weights must be True for the hook to receive weights; nn.Transformer layers call
# MultiheadAttention with need_weights=False, so re-run that layer's attention explicitly.
layer = model.transformer.decoder.layers[-1]
orig_forward = layer.multihead_attn.forward

def forward_with_weights(*args, **kwargs):
    kwargs["need_weights"] = True
    kwargs["average_attn_weights"] = True
    return orig_forward(*args, **kwargs)

layer.multihead_attn.forward = forward_with_weights
handle = layer.multihead_attn.register_forward_hook(_hook)

lm, tok = val_ds[0]
src = lm.unsqueeze(0).to(DEVICE)
mask = torch.ones(1, lm.shape[0], dtype=torch.bool, device=DEVICE)
tgt_in = tok[:-1].unsqueeze(0).to(DEVICE)
model.eval()
with torch.no_grad():
    model(src, mask, tgt_in, torch.ones_like(tgt_in, dtype=torch.bool))

handle.remove()
layer.multihead_attn.forward = orig_forward

if "w" in captured and captured["w"] is not None:
    attn = captured["w"][0].cpu().numpy()
    labels = [id_to_token.get(i, UNK) for i in tgt_in[0].tolist()]
    fig, ax = plt.subplots(figsize=(10, max(3, 0.35 * len(labels))))
    im = ax.imshow(attn, aspect="auto", cmap="viridis")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel("input frame"); ax.set_title(f'Cross-attention | ref: "{val_ds.df.iloc[0]["english_text"]}"')
    plt.colorbar(im, ax=ax); plt.tight_layout()
    plt.savefig("artifacts/decoder_attention.png", dpi=120); plt.show()
else:
    print("Attention weights not captured (PyTorch version may use a fused fast path); skipping this plot.")


**Token embedding space** — PCA of the decoder's learned embeddings for frequent words (diagnostic only).

In [ ]:
from sklearn.decomposition import PCA

top = [w for w, _ in word_counts.most_common(60) if w in token_to_id]
ids = [token_to_id[w] for w in top]
emb = model.tgt_embed.weight[ids].detach().cpu().numpy()
xy = PCA(n_components=2).fit_transform(emb)

fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(xy[:, 0], xy[:, 1], alpha=0.6)
for (x, y), w in zip(xy, top):
    ax.annotate(w, (x, y), fontsize=8)
ax.set_title("PCA of decoder token embeddings")
plt.tight_layout(); plt.savefig("artifacts/embedding_pca.png", dpi=120); plt.show()


## Step 10 — Production bundle and inference API

1. **One versioned artifact** with weights, vocab, architecture config and metrics.
2. An **`ISLTranslator`** class: `from_bundle(path)` then `translate(landmarks)`.
3. **Metadata** (data size, best val loss, BLEU, timestamp) so the bundle describes itself.

Input to `translate()` must be preprocessed exactly like training (`src/landmarks/pose_features.py`: select Hands+Pose → interpolate → shoulder-normalize → flatten to `frames x 225`).

In [ ]:
%%writefile src/inference/translator.py
"""Inference wrapper for the BRIDGE-ISL Transformer: load a bundle, call translate()."""
import re
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn

PAD, BOS, EOS, UNK = "<PAD>", "<BOS>", "<EOS>", "<UNK>"


class PositionalEncoding(nn.Module):
    def __init__(self, dim, max_len=2000):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, dim, 2).float() * (-np.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class Seq2SeqTransformer(nn.Module):
    def __init__(self, feature_dim, vocab_size, d_model=256, nhead=4, num_encoder_layers=3,
                 num_decoder_layers=3, dim_feedforward=512, dropout=0.1, pad_id=0):
        super().__init__()
        self.src_proj = nn.Linear(feature_dim, d_model)
        self.tgt_embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_enc = PositionalEncoding(d_model)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead, num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, src, src_key_padding_mask, tgt_in, tgt_key_padding_mask):
        src_emb = self.pos_enc(self.src_proj(src))
        tgt_emb = self.pos_enc(self.tgt_embed(tgt_in))
        n = tgt_in.size(1)
        causal = torch.triu(torch.ones(n, n, device=src.device) * float("-inf"), diagonal=1)
        out = self.transformer(
            src_emb, tgt_emb,
            src_key_padding_mask=~src_key_padding_mask,
            tgt_key_padding_mask=~tgt_key_padding_mask,
            tgt_mask=causal,
        )
        return self.out(out)


@dataclass
class ISLTranslator:
    model: Seq2SeqTransformer
    token_to_id: dict
    id_to_token: dict
    device: str = "cpu"
    max_gen_len: int = 40
    metadata: dict = field(default_factory=dict)

    @classmethod
    def from_bundle(cls, bundle_path, device=None):
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        ckpt = torch.load(bundle_path, map_location=device)
        model = Seq2SeqTransformer(pad_id=ckpt["token_to_id"][PAD], **ckpt["model_config"]).to(device)
        model.load_state_dict(ckpt["model_state_dict"])
        model.eval()
        return cls(model=model, token_to_id=ckpt["token_to_id"],
                   id_to_token={int(k): v for k, v in ckpt["id_to_token"].items()},
                   device=device, metadata=ckpt.get("metadata", {}))

    @torch.no_grad()
    def translate(self, landmarks: np.ndarray) -> str:
        """landmarks: (frames, feature_dim) float32, preprocessed like training."""
        src = torch.from_numpy(landmarks).float().unsqueeze(0).to(self.device)
        mask = torch.ones(1, landmarks.shape[0], dtype=torch.bool, device=self.device)
        ids = torch.tensor([[self.token_to_id[BOS]]], device=self.device)
        for _ in range(self.max_gen_len):
            logits = self.model(src, mask, ids, torch.ones_like(ids, dtype=torch.bool))
            nxt = logits[0, -1].argmax().item()
            ids = torch.cat([ids, torch.tensor([[nxt]], device=self.device)], dim=1)
            if nxt == self.token_to_id[EOS]:
                break
        toks = [self.id_to_token.get(i, UNK) for i in ids[0].tolist()]
        return " ".join(t for t in toks if t not in (PAD, BOS, EOS, UNK))


In [ ]:
import datetime

bundle_metadata = {
    "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "train_samples": len(train_ds),
    "val_samples": len(val_ds),
    "epochs_run": len(history["train_loss"]),
    "best_val_loss": float(best_val_loss),
    "bleu_on_val_subset": float(bleu.score),
    "bleu_eval_examples": N_EVAL,
    "vocab_size": VOCAB_SIZE,
    "extract_limit_used": EXTRACT_LIMIT,
    "feature_components": ["POSE_LANDMARKS", "LEFT_HAND_LANDMARKS", "RIGHT_HAND_LANDMARKS"],
    "seed": SEED,
}

bundle_path = "artifacts/isl_transformer_production.pt"
torch.save({
    "model_state_dict": model.state_dict(),   # best checkpoint was reloaded after training
    "token_to_id": token_to_id,
    "id_to_token": id_to_token,
    "model_config": MODEL_CONFIG,
    "metadata": bundle_metadata,
}, bundle_path)
with open("artifacts/bundle_info.json", "w") as f:
    json.dump({"model_config": MODEL_CONFIG, "metadata": bundle_metadata}, f, indent=2)

shutil.copy("src/inference/translator.py", "artifacts/translator.py")
shutil.copy("src/landmarks/pose_features.py", "artifacts/pose_features.py")
shutil.copy("data/vocab.json", "artifacts/vocab.json")
print(json.dumps(bundle_metadata, indent=2))


In [ ]:
import zipfile

final_zip = shutil.make_archive("bridge_isl_production_bundle", "zip", root_dir="artifacts")
print(f"{final_zip} ({os.path.getsize(final_zip) / 1e6:.1f} MB)")
with zipfile.ZipFile(final_zip) as z:
    for n in z.namelist():
        print(" -", n)

from IPython.display import FileLink
FileLink(final_zip)


**To keep it:** **Save Version → Save & Run All**, then open the **Output** tab to download, or **New Dataset** to publish it for reuse.

### Using the bundle the way a service would

In [ ]:
import sys
sys.path.insert(0, "src/inference")
from translator import ISLTranslator

translator = ISLTranslator.from_bundle(bundle_path, device=DEVICE)
print("Bundle metadata:", translator.metadata)

lm, _ = val_ds[0]
print("Reference :", val_ds.df.iloc[0]["english_text"])
print("Translated:", translator.translate(lm.numpy()))


## What determines real quality (and what to do next)

- **Data:** raise `EXTRACT_LIMIT` across sessions (Step 5b persists progress). BLEU on a few thousand segments will stay low; word-level sentence translation from landmarks alone is genuinely hard.
- **Check before trusting output:** BLEU, the length-calibration plot (are outputs collapsing to very short or repeated phrases?) and the sample generations.
- **Production caveats:** real-time use also needs a MediaPipe front-end matching this preprocessing exactly, latency measurements, and evaluation on unseen signers/conditions, plus review by ISL-proficient people before any linguistic claims (per the project spec).
- **Next per spec:** face landmarks (Section 6/11), better tokenization (subword), then the reverse direction (Section 13).
